In [1]:
from collections import Counter, defaultdict
from math import log
import numpy as np

In [2]:
def entropy(y, base=2):
    count = Counter(y)
    ans = 0
    for freq in count.values():
        prob = freq / len(y)
        ans -= prob * log(prob, base)
    return ans

In [3]:
def conditional_entropy(x, y, base=2):
    freq_y_total = defaultdict(Counter)
    freq_x = Counter()
    for i in range(len(x)):
        freq_y_total[x[i]][y[i]] += 1
        freq_x[x[i]] +=1
    ans = 0
    for xi, freq_y_xi in freq_y_total.items():
        res = 0
        for freq in freq_y_xi.values():
            prob = freq / freq_x[xi]
            res -= prob * log(prob, base)
        ans += res * (freq_x[xi] / len(x))
    return ans


In [4]:
class DecisionTreeID3:

    class Node:
        def __init__(self, mark, ee, use_feature=None, children=None):
            if children is None:
                children = {}
            self.mark = mark
            self.use_feature = use_feature
            self.children = children
            self.ee = ee

        @property
        def is_leaf(self):
            return len(self.children) == 0

    def __init__(self, x, y, labels=None, base=2, epsilon=0, alpha=0.05):
        if labels is None:
            labels = ["feature{}".format(i + 1) for i in range(len(x[0]))]
        self.labels = labels
        self.base = base
        self.epsilon = epsilon
        self.alpha = alpha

        self.n = len(x[0])
        self.root = self._build(x, y, set(range(self.n)))
        self._pruning(self.root)

    def _build(self, x, y, spare_features_idx):

        freq_y = Counter(y)
        ee = entropy(y, base=self.base)

        if len(freq_y) == 1:
            return self.Node(y[0], ee)

        if not spare_features_idx:
            return self.Node(freq_y.most_common(1)[0][0], ee)

        best_feature_idx, best_gain = -1, 0
        for feature_idx in spare_features_idx:
            gain = self.information_gain(x, y, feature_idx)
            if gain > best_gain:
                best_feature_idx, best_gain = feature_idx, gain

        if best_gain <= self.epsilon:
            return self.Node(freq_y.most_common(1)[0][0], ee)

        node = self.Node(freq_y.most_common(1)[0][0], ee, use_feature=best_feature_idx)
        features = set()
        sub_x = defaultdict(list)
        sub_y = defaultdict(list)
        for i in range(len(x)):
            feature = x[i][best_feature_idx]
            features.add(feature)
            sub_x[feature].append(x[i])
            sub_y[feature].append(y[i])

        for feature in features:
            node.children[feature] = self._build(sub_x[feature], sub_y[feature],
                                                 spare_features_idx - {best_feature_idx})
        return node

    """
    For each node, first prune all its children recursively.
    Then compare:
    loss1: cost if this node is turned into a leaf (no children).
    loss2: cost if this node keeps its subtree.
    If loss1 < loss2, delete children and make this node a leaf.
    """
    def _pruning(self, node):
        if node.is_leaf:
            return 1, node.ee

        loss1 = node.ee + 1 * self.alpha

        num, ee = 1, 0
        for child in node.children.values():
            child_num, child_ee = self._pruning(child)
            num += child_num
            ee += child_ee
        loss2 = ee + num * self.alpha

        if loss1 < loss2:
            node.children = {}
            return 1, node.ee

        else:
            return num, ee

    def __repr__(self):

        def dfs(node, depth=0, value=""):
            if node.is_leaf:
                res.append(value + " -> " + node.mark)
            else:
                if depth > 0:
                    res.append(value + " :")
                for val, child in node.children.items():
                    dfs(child, depth + 1, "  " * depth + self.labels[node.use_feature] + " = " + val)

        res = []
        dfs(self.root)
        return "\n".join(res)

    def information_gain(self, x, y, idx):
        """计算信息增益"""
        return entropy(y, base=self.base) - conditional_entropy([x[i][idx] for i in range(len(x))], y, base=self.base)


In [5]:
class DecisionTreeC45WithPruning(DecisionTreeID3):
    def information_gain(self, x, y, idx):
        return super().information_gain(x, y, idx) / entropy([x[i][idx] for i in range(len(x))], base=self.base)

In [6]:
X, Y = [np.array([["青年", "否", "否", "一般"],
                      ["青年", "否", "否", "好"],
                      ["青年", "是", "否", "好"],
                      ["青年", "是", "是", "一般"],
                      ["青年", "否", "否", "一般"],
                      ["中年", "否", "否", "一般"],
                      ["中年", "否", "否", "好"],
                      ["中年", "是", "是", "好"],
                      ["中年", "否", "是", "非常好"],
                      ["中年", "否", "是", "非常好"],
                      ["老年", "否", "是", "非常好"],
                      ["老年", "否", "是", "好"],
                      ["老年", "是", "否", "好"],
                      ["老年", "是", "否", "非常好"],
                      ["老年", "否", "否", "一般"]]),
            np.array(["否", "否", "是", "是", "否",
                      "否", "否", "是", "是", "是",
                      "是", "是", "是", "是", "否"])]

print(DecisionTreeID3(X, Y, labels=["年龄", "有工作", "有自己的房子", "信贷情况"], alpha=0.2))
print(DecisionTreeID3(X, Y, labels=["年龄", "有工作", "有自己的房子", "信贷情况"], alpha=0.3))

有自己的房子 = 否 :
  有工作 = 否 -> 否
  有工作 = 是 -> 是
有自己的房子 = 是 -> 是
 -> 是


In [7]:
decision_tree = DecisionTreeC45WithPruning(X, Y, labels=["年龄", "有工作", "有自己的房子", "信贷情况"])
print(decision_tree)

有自己的房子 = 否 :
  有工作 = 否 -> 否
  有工作 = 是 -> 是
有自己的房子 = 是 -> 是
